# Referencje i kopie — zagnieżdżone struktury obiektów

Wyobraźmy sobie serwis, w którym użytkownicy zapisują ulubione trasy spacerowe, biegowe lub rowerowe. Trasa ma nazwę i przebieg opisany listą punktów. Użytkownik może skopiować ulubione trasy innej osoby, a następnie dostosować je do swoich potrzeb. Chcemy, żeby późniejsze zmiany jego kopii nie wpływały na trasy źródłowe.

Zaczniemy od zbudowania takiej struktury. Następnie sprawdzimy, które jej elementy są wspólne, i porównamy przypisanie, płytką kopię i głęboką kopię. Definicje i kolejne operacje są w osobnych komórkach, żeby można było wykonywać je krok po kroku.

## Punkty, trasy i użytkownicy

Klasa `Wspolrzedne` przechowuje długość i szerokość geograficzną punktu.

In [ ]:
class Wspolrzedne:
    def __init__(self, dlugosc, szerokosc):
        self.dlugosc = dlugosc
        self.szerokosc = szerokosc

    def __repr__(self):
        return f"Wspolrzedne({self.dlugosc!r}, {self.szerokosc!r})"

### Reprezentacja diagnostyczna

Metoda [`__repr__`](https://docs.python.org/3/reference/datamodel.html#object.__repr__) powinna, jeśli to możliwe, zwracać wyrażenie pozwalające odtworzyć obiekt o tej samej wartości. Jeśli nie jest to możliwe, dokumentacja zaleca opis w nawiasach ostrych `<...>`.

Na przykład obiekt pliku zwrócony przez [`open`](https://docs.python.org/3/library/functions.html#open) może mieć taką reprezentację (przykładowy wynik):

```text
<_io.TextIOWrapper name='dane.csv' mode='r' encoding='utf-8'>
```

Nie jest to wywołanie konstruktora, ale opis zawierający nazwę pliku, tryb otwarcia i kodowanie.

W naszych klasach `Trasa` i `Uzytkownik` konstruktory nie przyjmują całego stanu, który chcemy pokazać. Dlatego ich reprezentacje zapiszemy jako opisy w nawiasach ostrych.

Trasa przechowuje nazwę i listę punktów. Metoda `dodaj_punkt` dopisuje otrzymany punkt do listy — nie kopiuje go.

In [ ]:
class Trasa:
    def __init__(self, nazwa):
        self.nazwa = nazwa
        self.przebieg = []

    def dodaj_punkt(self, punkt):
        self.przebieg.append(punkt)

    def __repr__(self):
        return f"<Trasa nazwa={self.nazwa!r}, przebieg={self.przebieg!r}>"

Użytkownik ma listę ulubionych tras. Na razie zdefiniujemy tylko dodawanie trasy. Również ta operacja nie tworzy kopii.

Metody `__repr__` pokazują stan zagnieżdżonych obiektów, żeby łatwiej było go obserwować. Dla `Trasa` i `Uzytkownik` są to opisy diagnostyczne w nawiasach ostrych.

In [ ]:
class Uzytkownik:
    def __init__(self, imie, nazwisko):
        self.imie = imie
        self.nazwisko = nazwisko
        self.ulubione_trasy = []

    def dodaj_ulubiona(self, trasa):
        self.ulubione_trasy.append(trasa)

    def __repr__(self):
        return (
            f"<Uzytkownik imie={self.imie!r}, nazwisko={self.nazwisko!r}, "
            f"ulubione_trasy={self.ulubione_trasy!r}>"
        )

In [ ]:
dom = Wspolrzedne(22.55, 51.24)
praca = Wspolrzedne(22.57, 51.25)

In [ ]:
trasa = Trasa("Z domu do pracy")

In [ ]:
trasa.dodaj_punkt(dom)
trasa.dodaj_punkt(praca)

In [ ]:
trasa

In [ ]:
jan = Uzytkownik("Jan", "Kowalski")

In [ ]:
jan.dodaj_ulubiona(trasa)

In [ ]:
jan

## Dostęp do zagnieżdżonych danych

Możemy przechodzić od użytkownika do trasy, od trasy do punktu, a następnie do pola tego punktu:

In [ ]:
jan.ulubione_trasy[0]

In [ ]:
jan.ulubione_trasy[0].przebieg

In [ ]:
jan.ulubione_trasy[0].przebieg[1]

In [ ]:
jan.ulubione_trasy[0].przebieg[1].dlugosc

To nie są kopie. Sprawdźmy, że na liście znajduje się ten sam obiekt punktu, który przekazaliśmy do metody:

In [ ]:
jan.ulubione_trasy[0].przebieg[0] is dom

In [ ]:
jan.ulubione_trasy[0].przebieg[0].dlugosc = 22.56

In [ ]:
dom

Możemy też użyć tego samego punktu na innej trasie. Zmiana jego współrzędnych będzie wtedy widoczna na obu trasach.

In [ ]:
anna = Uzytkownik("Anna", "Nowak")
dom_anny = Wspolrzedne(22.60, 51.26)
trasa_anny = Trasa("Z domu Anny do pracy")

In [ ]:
trasa_anny.dodaj_punkt(dom_anny)
trasa_anny.dodaj_punkt(praca)
anna.dodaj_ulubiona(trasa_anny)

In [ ]:
jan.ulubione_trasy[0].przebieg[1].dlugosc = 22.58

In [ ]:
anna.ulubione_trasy[0].przebieg[1]

## Dlaczego samo przypisanie nie wystarcza?

Anna chce teraz skopiować ulubione trasy Jana. Utworzymy ją ponownie, żeby zacząć z pustą listą ulubionych.

In [ ]:
anna = Uzytkownik("Anna", "Nowak")

In [ ]:
anna.ulubione_trasy = jan.ulubione_trasy

In [ ]:
anna.ulubione_trasy is jan.ulubione_trasy

In [ ]:
anna.dodaj_ulubiona(Trasa("Spacer Anny"))

In [ ]:
jan.ulubione_trasy

Samo przypisanie spowodowało współdzielenie listy. Dodanie trasy przez Annę zmieniło również listę widoczną u Jana. Przywróćmy osobną listę Jana z jego pierwotną trasą przed następną próbą.

In [ ]:
jan.ulubione_trasy = [trasa]
anna = Uzytkownik("Anna", "Nowak")

## Płytka kopia listy

Skopiujemy listę metodą [`copy`](https://docs.python.org/3/library/copy.html). Powstanie nowa lista, ale jej elementami będą te same obiekty tras.

In [ ]:
anna.ulubione_trasy = jan.ulubione_trasy.copy()

In [ ]:
anna.ulubione_trasy is jan.ulubione_trasy

In [ ]:
anna.dodaj_ulubiona(Trasa("Spacer Anny"))

In [ ]:
len(jan.ulubione_trasy), len(anna.ulubione_trasy)

Dodawanie tras jest już niezależne. Spróbujmy jednak zmienić nazwę skopiowanej trasy:

In [ ]:
anna.ulubione_trasy[0].nazwa = "Droga Anny do pracy"

In [ ]:
jan.ulubione_trasy[0].nazwa

In [ ]:
anna.ulubione_trasy[0] is jan.ulubione_trasy[0]

## Płytka kopia samej trasy

Skopiowanie samej trasy również nie rozwiązuje całego problemu. Powstaje nowy obiekt `Trasa`, ale jego pole `przebieg` wskazuje tę samą listę punktów.

In [ ]:
import copy

In [ ]:
kopia_trasy = copy.copy(trasa)

In [ ]:
kopia_trasy is trasa

In [ ]:
kopia_trasy.przebieg is trasa.przebieg

Nawet osobne skopiowanie listy punktów pozostawia wspólne obiekty współrzędnych:

In [ ]:
kopia_trasy.przebieg = trasa.przebieg.copy()

In [ ]:
kopia_trasy.przebieg is trasa.przebieg

In [ ]:
kopia_trasy.przebieg[0] is trasa.przebieg[0]

## Głęboka kopia tras

Potrzebujemy osobnych kopii tras, ich list przebiegu i punktów. Użyjemy [`deepcopy`](https://docs.python.org/3/library/copy.html#copy.deepcopy).

Dodajmy metodę `kopiuj_ulubione`. Dopisuje ona kopie do istniejącej listy użytkownika, zamiast ją zastępować. Nie usuwa duplikatów — ponowne wywołanie dopisze kolejne kopie.

Poniższa komórka zawiera całą nową definicję klasy. Po jej wykonaniu utworzymy nowych użytkowników, żeby korzystali z tej definicji.

In [ ]:
class Uzytkownik:
    def __init__(self, imie, nazwisko):
        self.imie = imie
        self.nazwisko = nazwisko
        self.ulubione_trasy = []

    def dodaj_ulubiona(self, trasa):
        self.ulubione_trasy.append(trasa)

    def __repr__(self):
        return (
            f"<Uzytkownik imie={self.imie!r}, nazwisko={self.nazwisko!r}, "
            f"ulubione_trasy={self.ulubione_trasy!r}>"
        )

    def kopiuj_ulubione(self, inny):
        self.ulubione_trasy.extend(copy.deepcopy(inny.ulubione_trasy))

In [ ]:
dom = Wspolrzedne(22.55, 51.24)
praca = Wspolrzedne(22.57, 51.25)
trasa = Trasa("Z domu do pracy")
trasa.dodaj_punkt(dom)
trasa.dodaj_punkt(praca)

In [ ]:
jan = Uzytkownik("Jan", "Kowalski")
jan.dodaj_ulubiona(trasa)
anna = Uzytkownik("Anna", "Nowak")

In [ ]:
anna.dodaj_ulubiona(Trasa("Spacer Anny"))

In [ ]:
anna.kopiuj_ulubione(jan)

In [ ]:
anna.ulubione_trasy

Zmieńmy kolejno nazwę skopiowanej trasy, współrzędne jednego punktu i liczbę punktów. Po każdej operacji sprawdźmy oryginał.

In [ ]:
kopia = anna.ulubione_trasy[1]

In [ ]:
kopia.nazwa = "Droga Anny do pracy"

In [ ]:
jan.ulubione_trasy[0].nazwa

In [ ]:
kopia.przebieg[0].dlugosc = 22.60

In [ ]:
jan.ulubione_trasy[0].przebieg[0]

In [ ]:
kopia.dodaj_punkt(Wspolrzedne(22.61, 51.26))

In [ ]:
len(jan.ulubione_trasy[0].przebieg), len(kopia.przebieg)

In [ ]:
jan.ulubione_trasy

In [ ]:
anna.ulubione_trasy

Anna może teraz zmieniać skopiowaną trasę bez zmieniania trasy Jana. W tym przykładzie [`deepcopy`](https://docs.python.org/3/library/copy.html#copy.deepcopy) utworzyło kopie zagnieżdżonych obiektów mutowalnych. Nie potrzebujemy osobnych kopii niemutowalnych napisów i liczb, żeby uzyskać tę niezależność.

### Głębokie kopiowanie bez modułu [`copy`](https://docs.python.org/3/library/copy.html)

Dla znanej nam struktury możemy sami utworzyć wszystkie potrzebne obiekty. Poniższa funkcja tworzy nową listę tras. Dla każdej trasy tworzy nowy obiekt `Trasa`, a dla każdego punktu — nowy obiekt `Wspolrzedne`. Nie używa modułu [`copy`](https://docs.python.org/3/library/copy.html).

In [ ]:
def kopiuj_trasy_recznie(trasy):
    kopie = []
    for trasa in trasy:
        nowa_trasa = Trasa(trasa.nazwa)
        for punkt in trasa.przebieg:
            nowy_punkt = Wspolrzedne(punkt.dlugosc, punkt.szerokosc)
            nowa_trasa.dodaj_punkt(nowy_punkt)
        kopie.append(nowa_trasa)
    return kopie

Taką funkcją można zastąpić wywołanie [`deepcopy`](https://docs.python.org/3/library/copy.html#copy.deepcopy) w metodzie `kopiuj_ulubione`:

```python
self.ulubione_trasy.extend(kopiuj_trasy_recznie(inny.ulubione_trasy))
```

Wypróbujmy tę samą operację na nowym użytkowniku:

In [ ]:
ola = Uzytkownik("Ola", "Lis")

In [ ]:
ola.ulubione_trasy.extend(kopiuj_trasy_recznie(jan.ulubione_trasy))

In [ ]:
ola.ulubione_trasy[0].przebieg[0].dlugosc = 22.70

In [ ]:
ola.ulubione_trasy[0].przebieg[0]

In [ ]:
jan.ulubione_trasy[0].przebieg[0]

Punkt oryginalnej trasy nie został zmieniony. Funkcja kopiuje wszystkie mutowalne poziomy tej konkretnej struktury, ale nie jest ogólnym zamiennikiem [`deepcopy`](https://docs.python.org/3/library/copy.html#copy.deepcopy).

Jeśli ten sam punkt występuje na dwóch trasach, pętla utworzy dwie osobne kopie. Nie zapamiętuje już skopiowanych obiektów. [`deepcopy`](https://docs.python.org/3/library/copy.html#copy.deepcopy) zachowuje takie powiązania.

### Wspólne punkty wewnątrz kopii

Jeśli dwie trasy Jana korzystają z tego samego punktu, [`deepcopy`](https://docs.python.org/3/library/copy.html#copy.deepcopy) podczas jednego wywołania tworzy jedną kopię tego punktu. Obie skopiowane trasy będą ją współdzielić. To nadal osobny punkt względem oryginału Jana.

In [ ]:
druga_trasa = Trasa("Powrót z pracy")
druga_trasa.dodaj_punkt(praca)
druga_trasa.dodaj_punkt(dom)
jan.dodaj_ulubiona(druga_trasa)

In [ ]:
ewa = Uzytkownik("Ewa", "Lis")
ewa.kopiuj_ulubione(jan)

In [ ]:
ewa.ulubione_trasy[0].przebieg[1] is ewa.ulubione_trasy[1].przebieg[0]

In [ ]:
ewa.ulubione_trasy[0].przebieg[1] is praca

Osiągnęliśmy teraz całkowitą niezależność tras między użytkownikami.

To, które obiekty powinny być wspólne, wynika z konkretnych potrzeb programu.